In [2]:
"""
Download 2026 US weather data from Open-Meteo, gridded across the continental US.

Requires: pip install requests
Run: python download_us_weather_2026.py

Output: us_weather_2026_grid.csv with columns:
  latitude, longitude, date, temperature_2m_max, temperature_2m_min, precipitation_sum,
  wind_speed_10m_max, sunshine_duration

Notes:
- Open-Meteo's free tier allows up to 10,000 calls/day, no API key needed.
- This script uses a 1-degree grid over the continental US bounding box
  (lat 25-49, lon -125 to -67) => ~900 points. Ocean points will simply
  return marine-adjacent weather (harmless, easy to filter out later by
  lat/lon if you only want strict land coverage).
- A ~0.2s delay between calls keeps you well under rate limits; the full
  run takes roughly 15-20 minutes for ~900 points x ~180 days each.
- To narrow scope (faster/smaller), reduce GRID_STEP (e.g. 2.0 for a
  coarser grid) or shrink the bounding box below.
"""

import csv
import time
import requests

# ---- Configuration ----
LAT_MIN, LAT_MAX = 25.0, 49.0
LON_MIN, LON_MAX = -125.0, -67.0
GRID_STEP = 1.0  # degrees; increase to 2.0 or 3.0 for a faster/coarser run

START_DATE = "2026-01-01"
END_DATE = "2026-07-01"  # adjust as needed; Open-Meteo will cap at latest available date

DAILY_VARS = "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunshine_duration"
OUTPUT_FILE = "us_weather_2026_grid.csv"
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# ---- Build grid points ----
def frange(start, stop, step):
    vals = []
    v = start
    while v <= stop + 1e-9:
        vals.append(round(v, 2))
        v += step
    return vals

lats = frange(LAT_MIN, LAT_MAX, GRID_STEP)
lons = frange(LON_MIN, LON_MAX, GRID_STEP)
grid_points = [(lat, lon) for lat in lats for lon in lons]

print(f"Total grid points: {len(grid_points)}")

# ---- Fetch and write ----
with open(OUTPUT_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["latitude", "longitude", "date", "temperature_2m_max",
                      "temperature_2m_min", "precipitation_sum",
                      "wind_speed_10m_max", "sunshine_duration"])

    for i, (lat, lon) in enumerate(grid_points, 1):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "daily": DAILY_VARS,
            "timezone": "auto",
        }
        try:
            resp = requests.get(API_URL, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            daily = data.get("daily", {})
            dates = daily.get("time", [])
            tmax = daily.get("temperature_2m_max", [])
            tmin = daily.get("temperature_2m_min", [])
            precip = daily.get("precipitation_sum", [])
            wind = daily.get("wind_speed_10m_max", [])
            sunshine = daily.get("sunshine_duration", [])

            for d, mx, mn, p, w, s in zip(dates, tmax, tmin, precip, wind, sunshine):
                writer.writerow([lat, lon, d, mx, mn, p, w, s])

            print(f"[{i}/{len(grid_points)}] lat={lat}, lon={lon} -> {len(dates)} days OK")

        except Exception as e:
            print(f"[{i}/{len(grid_points)}] lat={lat}, lon={lon} -> FAILED: {e}")

        time.sleep(0.2)  # be polite to the free API

print(f"\nDone. Data written to {OUTPUT_FILE}")


Total grid points: 1475
[1/1475] lat=25.0, lon=-125.0 -> 182 days OK
[2/1475] lat=25.0, lon=-124.0 -> 182 days OK
[3/1475] lat=25.0, lon=-123.0 -> 182 days OK
[4/1475] lat=25.0, lon=-122.0 -> 182 days OK
[5/1475] lat=25.0, lon=-121.0 -> 182 days OK
[6/1475] lat=25.0, lon=-120.0 -> 182 days OK
[7/1475] lat=25.0, lon=-119.0 -> 182 days OK
[8/1475] lat=25.0, lon=-118.0 -> 182 days OK
[9/1475] lat=25.0, lon=-117.0 -> 182 days OK
[10/1475] lat=25.0, lon=-116.0 -> 182 days OK
[11/1475] lat=25.0, lon=-115.0 -> 182 days OK
[12/1475] lat=25.0, lon=-114.0 -> 182 days OK
[13/1475] lat=25.0, lon=-113.0 -> 182 days OK
[14/1475] lat=25.0, lon=-112.0 -> 182 days OK
[15/1475] lat=25.0, lon=-111.0 -> 182 days OK
[16/1475] lat=25.0, lon=-110.0 -> 182 days OK
[17/1475] lat=25.0, lon=-109.0 -> 182 days OK
[18/1475] lat=25.0, lon=-108.0 -> 182 days OK
[19/1475] lat=25.0, lon=-107.0 -> 182 days OK
[20/1475] lat=25.0, lon=-106.0 -> 182 days OK
[21/1475] lat=25.0, lon=-105.0 -> 182 days OK
[22/1475] lat=25.0,